# GenAI Workshop
## Lesson 2: Prompt Engineering

This lesson is intended to improve your prompt engineering skills.

During this lesson you will learn how to ...
* use a system prompt to define model behaviour
* extend system prompt to create workflows
* specify output format



### Set up the environment 

Before we can start, we have to setup the environment and several default values for model name, model parameter and prompts.  

In [1]:
// Import required libraries for HTTP client and JSON handling
@file:DependsOn("com.squareup.okhttp3:okhttp:4.12.0")
@file:DependsOn("com.google.code.gson:gson:2.10.1")

import okhttp3.*
import okhttp3.MediaType.Companion.toMediaType
import okhttp3.RequestBody.Companion.toRequestBody
import com.google.gson.Gson
import com.google.gson.JsonObject
import com.google.gson.JsonArray
import com.google.gson.JsonParser
import java.nio.file.Files
import java.nio.file.Paths
import java.util.Base64

// Check runtime environment
val COLAB = System.getenv("COLAB_RELEASE_TAG") != null
if (COLAB) {
    println("Running on COLAB environment.")
} else {
    println("WARNING: Running on LOCAL environment.")
}

In [2]:
// Data paths
val PROCESSED_DATA_PATH = "books.db"
val BOOK_DB = PROCESSED_DATA_PATH

In [3]:
// Initialize Google GenAI Client API with GOOGLE_API_KEY
val GOOGLE_API_KEY = System.getenv("GOOGLE_API_KEY") 
    ?: throw IllegalStateException("GOOGLE_API_KEY environment variable not set")

println("GOOGLE_API_KEY set with a length of ${GOOGLE_API_KEY.length}")

GOOGLE_API_KEY set with a length of 39


### Definition of convenient functions  

The following methods will simplify working with the GEMINI genai model.
For details see function documentation.   

In [4]:
// Set default values for model, model parameters and prompt
val DEFAULT_MODEL = "gemini-2.0-flash-exp"
val DEFAULT_CONFIG_TEMPERATURE = 0.9
val DEFAULT_CONFIG_TOP_K = 1
val DEFAULT_CONFIG_MAX_OUTPUT_TOKENS = 2000
val DEFAULT_SYSTEM_PROMPT = "You are a friendly assistant"
val DEFAULT_USER_PROMPT = " "

// HTTP client
val client = OkHttpClient()
val gson = Gson()
val JSON = "application/json; charset=utf-8".toMediaType()

// Data class for API response
data class GeminiResponse(val text: String, val fullResponse: JsonObject)

// Initialize history with book database content
val bookDbPath = Paths.get(BOOK_DB)
val bookDbText = if (Files.exists(bookDbPath)) {
    String(Files.readAllBytes(bookDbPath), Charsets.UTF_8)
} else {
    println("Warning: Book database not found at $BOOK_DB")
    ""
}

// History stored as JSON objects
var history = mutableListOf<JsonObject>()

// Initialize with book database
fun initHistory() {
    history.clear()
    if (bookDbText.isNotEmpty()) {
        val initialContent = JsonObject().apply {
            addProperty("role", "user")
            add("parts", JsonArray().apply {
                add(JsonObject().apply {
                    addProperty("text", "Here is the book database:\n$bookDbText")
                })
            })
        }
        history.add(initialContent)
    }
}

// Initialize on first load
initHistory()

/**
 * Clear the history of the conversation.
 */
fun clearHistory() {
    initHistory()
    println("History cleared.")
}

/**
 * Call the GenAI model via REST API and return the response.
 * 
 * @param userPrompt The prompt to send to the model
 * @param systemPrompt The system prompt to use
 * @param modelName The name of the model to use
 * @param verbose If true, print the response
 * @return The response from the model
 */
fun generateBookstoreBotCompletion(
    userPrompt: String,
    systemPrompt: String = DEFAULT_SYSTEM_PROMPT,
    modelName: String = DEFAULT_MODEL,
    verbose: Boolean = false
): GeminiResponse {
    // Add user message to history
    val userContent = JsonObject().apply {
        addProperty("role", "user")
        add("parts", JsonArray().apply {
            add(JsonObject().apply {
                addProperty("text", userPrompt)
            })
        })
    }
    history.add(userContent)
    
    // Build request body
    val requestBody = JsonObject().apply {
        // System instruction
        add("system_instruction", JsonObject().apply {
            add("parts", JsonArray().apply {
                add(JsonObject().apply {
                    addProperty("text", systemPrompt)
                })
            })
        })
        
        // Contents (history)
        add("contents", JsonArray().apply {
            history.forEach { add(it) }
        })
        
        // Generation config
        add("generationConfig", JsonObject().apply {
            addProperty("temperature", DEFAULT_CONFIG_TEMPERATURE)
            addProperty("topK", DEFAULT_CONFIG_TOP_K)
            addProperty("maxOutputTokens", DEFAULT_CONFIG_MAX_OUTPUT_TOKENS)
        })
    }
    
    // Make HTTP request
    val url = "https://generativelanguage.googleapis.com/v1beta/models/$modelName:generateContent?key=$GOOGLE_API_KEY"
    val request = Request.Builder()
        .url(url)
        .post(gson.toJson(requestBody).toRequestBody(JSON))
        .build()
    
    client.newCall(request).execute().use { response ->
        if (!response.isSuccessful) {
            throw RuntimeException("API call failed: ${response.code} - ${response.body?.string()}")
        }
        
        val responseBody = response.body?.string() ?: throw RuntimeException("Empty response")
        val jsonResponse = JsonParser.parseString(responseBody).asJsonObject
        
        // Extract text from response
        val candidates = jsonResponse.getAsJsonArray("candidates")
        if (candidates == null || candidates.size() == 0) {
            throw RuntimeException("No candidates in response: $responseBody")
        }
        
        val firstCandidate = candidates[0].asJsonObject
        val content = firstCandidate.getAsJsonObject("content")
        val parts = content.getAsJsonArray("parts")
        val text = parts[0].asJsonObject.get("text").asString
        
        // Add assistant response to history
        history.add(content)
        
        if (verbose) {
            println("User Prompt: $userPrompt")
            println("Assistant Response: $text")
        }
        
        return GeminiResponse(text, jsonResponse)
    }
}

In [5]:
/**
 * Prints out the completion result.
 * 
 * @param completionResult The completion response
 * @param full Whether to print all details or only the text
 */
fun printCompletionResult(completionResult: GeminiResponse, full: Boolean = false) {
    println("\n#### ANSWER of genAI model:\n")
    if (full) {
        println(gson.toJson(completionResult.fullResponse))
    } else {
        println(completionResult.text)
    }
}

In [6]:
/**
 * Orders a book by its ISBN number.
 * 
 * @param isbn The ISBN number of the book to order
 */
fun orderBook(isbn: String) {
    println("Ordering book with ISBN: $isbn")
    
    // Print "." every 0.5 seconds
    repeat(5) {
        print(".")
        Thread.sleep(500)
    }
    println()
    
    if (isbn == "978-3-51593-12345-6") {
        println("Success: You ordered 'A Study in Scarlet'!")
        println("You completed this exercise successfully!")
    } else {
        println("Error: Unknown ISBN number.")
    }
}

### Important notice: History
In this (and only this) exercise, we implemented a history feature, so that the conversation can keep on going. If you mess up, use the following function to clear the history.

In [ ]:
// Clean the chat history and put the books.db into the chat history as the first entry
clearHistory()

### Exercise 01: Flipped interaction
Your task is to create a simple bookstore chatbot, which is able to gather more information about a book from a customer, if the customer did not provide enough information.

Imagine the following situation:
A customer of a bookstore wants to buy a book, but provides almost no information.  
The customer might ask an employee: *"I'm looking for this one book about a detective."*  
The bookstore employee needs more information in order to help the customer. The employee might ask: "I'm sure I can help you, but I need more information. Do you know the name of the detective or do you know more about the content of the book?"
  
The employee's reaction described here, should now be done by a chatbot. Your task is to provide a system prompt for this bot. 
The bot should:  
* Ask the customer for more information if the provided information is not enough for finding the book.

In [7]:
// Do not change the user prompt.
val userPrompt = "I'm looking for this one book about a detective."

// TODO: Define the system prompt of the bookstore bot using the "Flipped Interaction" pattern
val systemPrompt = """You are an employee in a book store. In order to advise the customer you first need to ask about the details what the 
customer actually really wants."""

// Test the chatbot
val response = generateBookstoreBotCompletion(userPrompt = userPrompt, systemPrompt = systemPrompt, verbose = false)
printCompletionResult(response)


#### ANSWER of genAI model:

Okay, I can help with that! To narrow things down, could you tell me a bit more about the book? For example:

*   **Do you remember the title or the author?** Even a partial name would be helpful.
*   **What kind of detective is it?** Is it a hard-boiled private eye, a police detective, an amateur sleuth, or something else?
*   **Do you know when it was published or when you read it?** This can help narrow down the possibilities.
*   **What is the book about?** For example, a crime, a missing person, etc.
*   **Do you know any distinguishing features of the book?** (e.g., cover art, genre, the setting of the book)
*   **Was it part of a series?**
*   **Is it a new book or an older one?**

The more information you can give me, the better chance I have of finding the right book for you!



### Exercise 02: Basic interaction
Your task is to create a simple chatbot, which is able to order a book based on the description of a book.  
Imagine the following situation:
A customer of a bookstore wants to buy a book, but does not remember its title.  
The customer might ask an employee: *"I'm looking for this book, where Sherlock Holmes and Watson meet the first time."*
In response an employee will use his/her extensive knowledge of books and say: "That's "A Study in Scarlet"! Should I place an order for this book?"
  
The employee's reaction described here, should now be done by a chatbot. Your task is to provide a system prompt for this bot. 
The bot should:  
* Respond to the customer by naming the book title, year of publication and a short summary of the book.
* In addition the bot should ask, if the book should be ordered.  

**Hints**: The creators of the bot (aka teacher of this workshop) already provided knowledge about books to the bot. You do not need to worry about this.


 

In [8]:
// Clear the chat history
clearHistory()

// Do not change the user prompt.
val userPrompt = "I'm looking for this book, where Sherlock Holmes and Watson meet the first time."

// TODO: Define the respective system prompt.
val systemPrompt = """you are an employee in a bookstore. if a customer is asking for a book but only has a rough idea for what he is looking for 
search the given books list for the book that seems most appropriate. Tell the customer the name of the book and ask the customer whether you should
order the book"""

// We provide some knowledge to the model about the book contents.
val response = generateBookstoreBotCompletion(userPrompt = userPrompt, systemPrompt = systemPrompt)
printCompletionResult(response)

History cleared.

#### ANSWER of genAI model:

Ah, you're looking for the story where Sherlock Holmes and Dr. Watson first meet! That would be "A Study in Scarlet." Would you like me to order that for you?



### Exercise 03: Extend the process
Extend the system prompt even further.
After the bot provided the information about the book and asks if it should be ordered, the *customer might say*: "Yes, please!" As an result, the bot will order the book. This process is finished by the bot replying with an object like {"isbn": "978-4-23050-12345-6"}. This object represents the payload for ordering the book using an API.
Your task is to extend the system prompt in order to achieve this behaviour.

In [9]:
// TODO: Define the respective system prompt.
val systemPrompt = """you are an employee in a bookstore. if a customer is asking for a book but only has a rough idea for what he is looking for 
search the given books list for the book that seems most appropriate. Tell the customer the name of the book and ask the customer whether you should
order the book. After the customer affirms that he wants it to be ordered, reply by giving the isbn-number in the format {\"isbn\": \"<isnb-number>\"}"""

In [10]:
// Clear the chat history
clearHistory()

// The customer is looking for a book.
val customerInitialPrompt = "I'm looking for this book, where Sherlock Holmes and Watson meet the first time."
println("Customer:\n$customerInitialPrompt\n")

// The bookstore bot answers the customer.
var response = generateBookstoreBotCompletion(userPrompt = customerInitialPrompt, systemPrompt = systemPrompt)
println("Bookstore bot:\n${response.text}\n")

// The customer wants to order the book.
val customerAnswer = "Yes, I like to order the book."
println("Customer:\n$customerAnswer\n")

// The bookstore bot answers the customer.
response = generateBookstoreBotCompletion(userPrompt = customerAnswer, systemPrompt = systemPrompt)
println("Bookstore bot:\n${response.text}\n")

History cleared.
Customer:
I'm looking for this book, where Sherlock Holmes and Watson meet the first time.

Bookstore bot:
Based on your description, you're likely looking for "A Study in Scarlet." It's the novel where Sherlock Holmes and Dr. Watson first meet and begin their partnership.

Would you like me to order it for you?


Customer:
Yes, I like to order the book.

Bookstore bot:
{\"isbn\": \"978-1503206857\"}




In [11]:
// TODO: Let's test, if the bot provided the correct isbn for the Sherlock Holmes book.
// Copy the isbn of the last bot response and assign the isbn variable below to the copied value.
// If you done it correctly, you should see a success message.
val isbn = "978-3-51593-12345-6"
orderBook(isbn = isbn)

Ordering book with ISBN: 978-3-51593-12345-6
.....
Success: You ordered 'A Study in Scarlet'!
You completed this exercise successfully!


### Exercise 04: Give choices
After the initial customer request, the bot should give the user another choice. In addition to ordering the book, the bot can provide an url to the ebook version of the book as well. Update the system prompt in order to achieve this.

In [12]:
// TODO Update the system prompt to include the option to provide an url to the ebook instead of ordering the book.
val systemPrompt = """
you are an employee in a bookstore. if a customer is asking for a book but only has a rough idea for what he is looking for 
search the given books list for the book that seems most appropriate. Tell the customer the name of the book and ask the customer 
whether you should
order the book or if she wants an ebook. If the user wants an ebook give her the url to the book from the data provided.
Or , if the user likes real paper books better, give her the isbn number in the format {\"isbn\": \"<isnb-number>\"}
"""

In [13]:
// Clear the chat history
clearHistory()

// The customer is looking for a book.
val customerInitialPrompt = "I'm looking for this book, where Sherlock Holmes and Watson meet the first time."
println("Customer:\n$customerInitialPrompt\n")

// The bookstore bot answers the customer.
var response = generateBookstoreBotCompletion(userPrompt = customerInitialPrompt, systemPrompt = systemPrompt)
println("Bookstore bot:\n${response.text}\n")

// The customer wants to have the link to the ebook.
val customerAnswer = "I love ebooks. Why should I order a book, if I get an ebook for free. Please provide the link!"
println("Customer:\n$customerAnswer\n")

// The bookstore bot answers the customer.
response = generateBookstoreBotCompletion(userPrompt = customerAnswer, systemPrompt = systemPrompt)
println("Bookstore bot:\n${response.text}\n")

History cleared.
Customer:
I'm looking for this book, where Sherlock Holmes and Watson meet the first time.

Bookstore bot:
Ah, you're looking for the story of how Sherlock Holmes and Dr. Watson first met! That would be "A Study in Scarlet".

Would you like me to order a physical copy for you, or would you prefer the ebook?


Customer:
I love ebooks. Why should I order a book, if I get an ebook for free. Please provide the link!

Bookstore bot:
Great choice! Here's the link to the ebook version of "A Study in Scarlet":

[https://www.gutenberg.org/ebooks/244](https://www.gutenberg.org/ebooks/244)




In [14]:
// Test if the old process of ordering a book still works

// Clear the chat history
clearHistory()

// The customer is looking for a book.
val customerInitialPrompt = "I'm looking for this book, where Sherlock Holmes and Watson meet the first time."
println("Customer:\n$customerInitialPrompt\n")

// The bookstore bot answers the customer.
var response = generateBookstoreBotCompletion(userPrompt = customerInitialPrompt, systemPrompt = systemPrompt)
println("Bookstore bot:\n${response.text}\n")

// The customer wants to order the book.
val customerAnswer = "Please order the book. I love spending money for stuff I can get for free!"
println("Customer:\n$customerAnswer\n")

// The bookstore bot answers the customer.
response = generateBookstoreBotCompletion(userPrompt = customerAnswer, systemPrompt = systemPrompt)
println("Bookstore bot:\n${response.text}\n")

History cleared.
Customer:
I'm looking for this book, where Sherlock Holmes and Watson meet the first time.

Bookstore bot:
Ah, you're looking for the story of how Sherlock Holmes and Dr. Watson first met! That would be "A Study in Scarlet" by Arthur Conan Doyle.

Would you prefer I order a physical copy of the book for you, or would you be interested in the eBook version?


Customer:
Please order the book. I love spending money for stuff I can get for free!

Bookstore bot:
Okay, I can order the physical book for you. Here is the ISBN:

```json
{
"isbn": "978-1404386364"
}
```


